In [ ]:
# ==== SETUP (run first): mount Drive, clone/pull repo, restore .npz ====
from pathlib import Path
import os, subprocess, sys, shutil

from google.colab import drive
drive.mount('/content/drive')

DRIVE_REPO = Path("/content/drive/MyDrive/music-analyzer_v2")
LOCAL_PARSED = Path("/content/parsed_dataset")
DRIVE_PARSED = DRIVE_REPO / "parsed_dataset"

if not DRIVE_REPO.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/ChristosPaterakisntua/music-analyzer-v2.git",
         str(DRIVE_REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(DRIVE_REPO), "pull"], check=True)

LOCAL_PARSED.mkdir(parents=True, exist_ok=True)

# env vars read by music_analyzer_v2.utils (set BEFORE importing the package)
os.environ["MA2_PARSED_DATASET"] = str(LOCAL_PARSED)
os.environ["MA2_CHECKPOINT_DIR"] = str(DRIVE_REPO / "checkpoints")
os.environ["MA2_SAVED_MODELS_DIR"] = str(DRIVE_REPO / "saved_models")
os.environ["PYTHONPATH"] = str(DRIVE_REPO / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, str(DRIVE_REPO / "src"))

# restore .npz + labels.json from Drive to the fast local disk
restored = 0
for source in sorted(DRIVE_PARSED.glob("*.npz")):
    target = LOCAL_PARSED / source.name
    if target.exists() and target.stat().st_size == source.stat().st_size:
        continue
    shutil.copy2(source, target)
    restored += 1
labels = DRIVE_PARSED / "labels.json"
if labels.exists():
    shutil.copy2(labels, LOCAL_PARSED / "labels.json")
print(f"restored {restored} .npz -> {LOCAL_PARSED}")


In [ ]:
from music_analyzer_v2.dataset import MusicDataLoader, MusicDataset
from music_analyzer_v2.models import MusicAnalyzerModel0, MusicAnalyzerModel1
from music_analyzer_v2.train import train
from music_analyzer_v2.utils import select_device

Device selection

<small>*expected: cuda*</small>

In [ ]:
device = select_device()
print(f"Device: {device}")

Dataset loading & normalization

In [ ]:
train_dataset = MusicDataset(split="train")
validation_dataset = MusicDataset(split="validation")

train_mean, train_std = train_dataset.fit_scalar_standardizer()
train_dataset.set_scalar_standardizer(train_mean, train_std)
validation_dataset.set_scalar_standardizer(train_mean, train_std)

Dataloaders (train & validation) loading the dataset

In [ ]:
train_dataloader = MusicDataLoader(
    dataset=train_dataset,
    device=device,
)

validation_dataloader = MusicDataLoader(
    dataset=validation_dataset,
    device=device,
    shuffle=False,
)

Initialization of the models

In [ ]:
model0 = MusicAnalyzerModel0().to(device)
model1 = MusicAnalyzerModel1().to(device)

Model 0 training

In [ ]:
train(
    model=model0,
    train_data_loader=train_dataloader,
    validation_data_loader=validation_dataloader,
    device=device,
    name="music_analyzer_model_0.pt",
)

Model 1 training

In [ ]:
train(
    model=model1,
    train_data_loader=train_dataloader,
    validation_data_loader=validation_dataloader,
    device=device,
    name="music_analyzer_model_1.pt",
)